In [0]:

def upsertToInvoices(invoices_df, batch_id):
    invoices_df.createOrReplaceTempView("invoices_df_temp_view")

    merge_statement = """
        MERGE INTO spark_stream.default.invoices_bz AS s
        USING invoices_df_temp_view AS t
        ON s.value = t.value
           AND s.timestamp = t.timestamp

        WHEN MATCHED THEN
            UPDATE SET *

        WHEN NOT MATCHED THEN
            INSERT *
    """

    invoices_df.sparkSession.sql(merge_statement)


In [0]:
class Bronze():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")
        self.BOOTSTRAP_SERVER = dbutils.secrets.get(scope="kafka-secrets",key="bootstrap-server")
        self.JAAS_MODULE = "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule"
        self.CLUSTER_API_KEY = dbutils.secrets.get(scope="kafka-secrets",key="cluster-api-key")
        self.CLUSTER_API_SECRET = dbutils.secrets.get(scope="kafka-secrets",key="cluster-api-secret")

    def getSchema(self):
        return """InvoiceNumber string
                , CreatedTime bigint
                , StoreID string
                , PosID string
                , CashierID string
                , CustomerType string
                , CustomerCardNo string
                , TotalAmount double
                , NumberOfItems bigint
                , PaymentMethod string
                , TaxableAmount double
                , CGST double
                , SGST double
                , CESS double
                , DeliveryType string
                , DeliveryAddress struct<AddressLine string
                , City string
                , ContactNumber string
                , PinCode string
                , State string>
                , InvoiceLineItems array<struct<ItemCode string
                , ItemDescription string
                , ItemPrice double
                , ItemQty bigint
                , TotalValue double>>
            """

    def ingestFromKafka(self, startingTime = 1):
        return ( 
                    spark.readStream
                        .format("kafka")            
                        .option("kafka.bootstrap.servers", self.BOOTSTRAP_SERVER)
                        .option("kafka.security.protocol", "SASL_SSL")
                        .option("kafka.sasl.mechanism", "PLAIN")
                        .option("kafka.sasl.jaas.config", f"{self.JAAS_MODULE} required username='{self.CLUSTER_API_KEY}' password='{self.CLUSTER_API_SECRET}';")
                        .option("subscribe", "invoices")
                        .option("maxOffsetsPerTrigger", 30)
                        .option("startingTimestamp", startingTime)
                        .load()
                )
        
    def getInvoices(self, kafka_df):
        from pyspark.sql.functions import cast, from_json
        return (
                    kafka_df.select(kafka_df.key.cast("string").alias("key"),
                            from_json(kafka_df.value.cast("string"), self.getSchema()).alias("value"),
                            "topic", "timestamp")
                )

    def process(self, startingTime = 1):
        print(f"Starting Bronze Stream...", end='')
        rawDF = self.ingestFromKafka(startingTime)
        invoicesDF = self.getInvoices(rawDF)
        sQuery =  ( 
                        invoicesDF.writeStream
                            .queryName("bronze-ingestion")
                            .foreachBatch(upsertToInvoices)
                            .option("checkpointLocation", f"{self.checkpoint_dir}/invoices_bz")
                            .outputMode("append")
                            .trigger(availableNow=True)
                            .start()          
                    ) 
        print("Done")
        return sQuery 